# Stage 16 — decoupled CLIP features + temporal-CTC + KenLM decoding (Kaggle T4)

Beat the baseline on **English** WiTA, reporting **lex** and **nonlex** CER separately. No monolithic video-VLM — vision and language are decoupled:

1. **Frozen CLIP ViT-B/16** → per-frame features, cached `[T,512]` per clip (ViT never runs in training).
2. **Compact BiLSTM + CTC** (V=27 standard CTC) on cached features — tiny model, large batches, T4-easy.
3. **KenLM beam search via pyctcdecode** — **word-LM (+lexicon) for lex**, **greedy/char-LM for nonlex** (a word LM hurts random strings). This is the main CER lever; runs on CPU.
4. **Temporal augmentation** on cached features (frame-drop / speed-perturb), label-preserving.

## Settings
- Accelerator: GPU T4 ×1; Internet: ON (CLIP + KenLM build).
- Attach `gaurs86/wita-full-english-122signers`.

## Discipline
Test evaluated exactly once at the end (marker-gated). Tune LM weights on val only.

## Cell 1 — install + clone

In [ ]:
import sys, subprocess

# CRITICAL: do NOT let pip downgrade numpy.  Kaggle ships numpy 2.x and its
# transformers/scipy are compiled against it.  An older pyctcdecode pins
# numpy<2; installing it normally downgrades numpy to 1.26.4 and then scipy/
# transformers break with "No module named 'numpy.strings'".
# Fix: install pyctcdecode with --no-deps (+ its real runtime dep pygtrie),
# and DO NOT reinstall transformers (Kaggle's already works with numpy 2.x).
def _pip(*pkgs, no_deps=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + (['--no-deps'] if no_deps else []) + list(pkgs)
    subprocess.run(cmd, check=True, capture_output=True)   # hide noisy resolver warnings

_pip('editdistance', 'kenlm', 'pygtrie')
_pip('pyctcdecode', no_deps=True)

import numpy, transformers, pyctcdecode
print('numpy', numpy.__version__, '(must be 2.x)  |  transformers', transformers.__version__,
      '|  pyctcdecode', getattr(pyctcdecode, '__version__', '?'))
assert numpy.__version__.startswith('2'), 'numpy got downgraded -> Restart kernel and re-run this cell'

import os, glob
import subprocess as _sp
_sp.run('rm -rf /kaggle/working/wita_v2', shell=True)
_sp.run("git clone -b stage13b-paper-replication "
        "'https://github.com/Gaurs86/WiTA-v2.git' '/kaggle/working/wita_v2'", shell=True, check=True)
sys.path.insert(0, '/kaggle/working/wita_v2')
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Cell 2 — locate data; set paths

In [ ]:
from stage16.common import find_data_root
DATA_ROOT = find_data_root()
CACHE_ROOT = '/kaggle/working/clip_feats'
CKPT_DIR  = '/kaggle/working/stage16_ctc'
LM_DIR    = '/kaggle/working/lm'
T_FRAMES  = 32   # CLIP per-frame at 32 frames -> CTC headroom for long words
print('DATA_ROOT =', DATA_ROOT)

## Cell 3 — extract + cache CLIP features (one-time, resumable)

~all 6 split/subsets. Frozen CLIP, fp16, batched. Skips existing `.npy`.

In [ ]:
from stage16.clip_feature_cache import extract_split
for split in ('val','test','train'):           # val/test first (small) to fail fast
    for subset in ('lex','nonlex'):
        extract_split(DATA_ROOT, CACHE_ROOT, split, subset, t=T_FRAMES, batch=64)

## Cell 4 — sanity: cached feature shape + model self-test

In [ ]:
import numpy as np
f = glob.glob(f'{CACHE_ROOT}/val/lex/*.npy')[0]
arr = np.load(f); print('cached feature:', arr.shape, arr.dtype, '(expect [<=32,512] float16)')
from stage16.temporal_ctc import _selftest
_selftest()   # builds model, 50-step overfit; loss should fall

## Cell 5 — train temporal CTC (BiLSTM) on cached features

In [ ]:
from stage16.temporal_ctc import train_ctc
best = train_ctc(DATA_ROOT, CACHE_ROOT, out_dir=CKPT_DIR,
                 backbone='bilstm', d_model=256, n_layers=3, dropout=0.3,
                 epochs=40, batch=64, lr=3e-4)
print('best val:', best)   # reports overall / lex / nonlex (greedy)

## Cell 6 — build KenLM (word 4-gram + char 6-gram) from TRAIN labels

Compiles KenLM's `lmplz`/`build_binary` (~5-8 min). Corpora are train-only (no leakage).

In [ ]:
from stage16.lm_decode import build_corpora
corp = build_corpora(DATA_ROOT, LM_DIR)
import os
if not os.path.isfile('/kaggle/working/kenlm/build/bin/lmplz'):
    !apt-get -qq install -y build-essential cmake libboost-all-dev libeigen3-dev >/dev/null 2>&1
    !cd /kaggle/working && git clone -q https://github.com/kpu/kenlm.git && \
        mkdir -p kenlm/build && cd kenlm/build && cmake .. -DCMAKE_BUILD_TYPE=Release >/dev/null && make -j4 >/dev/null 2>&1
LMPLZ='/kaggle/working/kenlm/build/bin/lmplz'; BUILDBIN='/kaggle/working/kenlm/build/bin/build_binary'
# Word LM (4-gram). --discount_fallback for tiny corpora.
!{LMPLZ} -o 4 --discount_fallback < {LM_DIR}/word_corpus.txt > {LM_DIR}/word.arpa 2>/dev/null
!{BUILDBIN} {LM_DIR}/word.arpa {LM_DIR}/word.bin 2>/dev/null
# Char LM (6-gram) for nonlex option.
!{LMPLZ} -o 6 --discount_fallback < {LM_DIR}/char_corpus.txt > {LM_DIR}/char.arpa 2>/dev/null
!{BUILDBIN} {LM_DIR}/char.arpa {LM_DIR}/char.bin 2>/dev/null
print('word.bin:', os.path.isfile(f'{LM_DIR}/word.bin'), ' char.bin:', os.path.isfile(f'{LM_DIR}/char.bin'))

## Cell 7 — VAL: greedy vs LM (tune alpha/beta here, NOT on test)

lex → word-LM (+lexicon); nonlex → greedy (word LM hurts random strings).

In [ ]:
import torch
from stage16.common import CharConverter
from stage16.lm_decode import load_model, make_decoders, evaluate
device='cuda' if torch.cuda.is_available() else 'cpu'
conv=CharConverter(); model=load_model(f'{CKPT_DIR}/best.pt', device)
# small alpha/beta sweep on val for the word LM
best=None
for alpha in (0.3,0.5,0.8):
    for beta in (0.5,1.5,3.0):
        wdec,cdec=make_decoders(conv, word_lm=f'{LM_DIR}/word.bin', char_lm=f'{LM_DIR}/char.bin',
                                lex_unigrams=corp['lex_words'], alpha_word=alpha, beta_word=beta)
        r=evaluate(model, DATA_ROOT, CACHE_ROOT, 'val', conv, device,
                   word_dec=wdec, char_dec=cdec, nonlex_mode='greedy')
        print(f'alpha={alpha} beta={beta}  lex greedy={r["greedy"]["lex"]:.4f} -> LM={r["lm"]["lex"]:.4f}'
              f'   overall {r["greedy"]["overall"]:.4f} -> {r["lm"]["overall"]:.4f}')
        if best is None or r['lm']['overall']<best[0]:
            best=(r['lm']['overall'],alpha,beta,r)
print('\nBEST val (alpha,beta):', best[1], best[2], '-> overall', round(best[0],4))

## Cell 8 — TEST: run ONCE with the val-best alpha/beta

In [ ]:
import os, json
MARKER=f'{CKPT_DIR}/.stage16_test_evaluated'
assert not os.path.exists(MARKER), 'Test already evaluated once. Delete marker only if intentional.'
_,A,Bp,_=best
wdec,cdec=make_decoders(conv, word_lm=f'{LM_DIR}/word.bin', char_lm=f'{LM_DIR}/char.bin',
                        lex_unigrams=corp['lex_words'], alpha_word=A, beta_word=Bp)
res=evaluate(model, DATA_ROOT, CACHE_ROOT, 'test', conv, device,
             word_dec=wdec, char_dec=cdec, nonlex_mode='greedy')
res['val_best_alpha_beta']=[A,Bp]
json.dump(res, open(f'{CKPT_DIR}/stage16_test.json','w'), indent=2, default=float)
open(MARKER,'w').write('done')
print('='*64)
print('  STAGE 16 TEST (CLIP+BiLSTM+CTC, word-LM lex / greedy nonlex)')
print('='*64)
print(f"  GREEDY  lex={res['greedy']['lex']:.4f}  nonlex={res['greedy']['nonlex']:.4f}  overall={res['greedy']['overall']:.4f}")
print(f"  +LM     lex={res['lm']['lex']:.4f}  nonlex={res['lm']['nonlex']:.4f}  overall={res['lm']['overall']:.4f}")
print(f"  paper   lex=0.281   nonlex=0.365   overall=0.2924")
print('='*64)

## Cell 9 — results table

Fill from the printed numbers for the writeup:

| Decoder | lex CER | nonlex CER | overall |
|---|---|---|---|
| Stage 16 greedy | … | … | … |
| Stage 16 + word-LM (lex) / greedy (nonlex) | … | … | … |
| Paper baseline | 0.281 | 0.365 | 0.2924 |

Note what helped: the word-LM should cut **lex** CER (closed-ish vocab); **nonlex** stays at greedy (a word LM would hurt random strings). If frozen CLIP features plateau, the ablation is to lightly unfreeze the top CLIP blocks (re-extract) or swap to DINOv2.